## Variáveis de ambiente (3) — Wyscout

`download_data.py` e `download_heatmaps.py` usam os mesmos três valores:

- `WYSCOUT_SEARCH_TOKEN` — token da API (query param `token`)
- `WYSCOUT_GROUP_ID` — ex.: `1432001`
- `WYSCOUT_SUBGROUP_ID` — ex.: `479792`

### No terminal (zsh/bash)

```bash
export WYSCOUT_SEARCH_TOKEN="…"
export WYSCOUT_GROUP_ID="…"
export WYSCOUT_SUBGROUP_ID="…"
```

### Neste notebook

Preenche os valores na célula seguinte **antes** de correr o download.

In [ ]:
import os
from pathlib import Path


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "download_data.py").is_file():
            return p
    raise FileNotFoundError(
        "Não encontrei a raiz do repo (scripts/download_data.py). "
        "Abre o notebook a partir de raumdeuterapp ou faz cd para essa pasta."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
print("Repo root:", ROOT)

# Preencher (ou já exportadas no shell antes de iniciar Jupyter)
os.environ["WYSCOUT_SEARCH_TOKEN"] = "34edd9fc7fc3bb05fd09472940371b53d043ef98"
os.environ["WYSCOUT_GROUP_ID"] = "1432001"
os.environ["WYSCOUT_SUBGROUP_ID"] = "479792"

## 1. Download — `download_data.py`

Descarrega resultados da pesquisa Wyscout para `data/players/wyscout/` (um CSV por liga/temporada). Há ~2 s entre pedidos HTTP.

In [ ]:
%run scripts/download_data.py

## 1.b Heatmaps Wyscout — `download_heatmaps.py`

Opcional: **depois** de existirem os parquets consolidados `data/players/all/{ano}_all_leagues.parquet` (gerados mais abaixo no pipeline). Usa os **mesmos** três tokens que na secção inicial.

Escreve `data/players/heatmaps/heatmaps_2025.parquet` e `heatmaps_2026.parquet` (GraphQL `playerHeatmap`, 5 workers em paralelo; re-correr só preenche pares em falta).

### No terminal (operacional — precisa das variáveis Wyscout)

```bash
WYSCOUT_SEARCH_TOKEN=… WYSCOUT_GROUP_ID=… WYSCOUT_SUBGROUP_ID=… \
  apps/api/.venv/bin/python scripts/download_heatmaps.py
```

Substitui `…` pelos valores (ou usa `export` nas três linhas e um `apps/api/.venv/bin/python scripts/download_heatmaps.py` sem prefixo).

### A partir deste notebook

Corre a célula seguinte **depois** da célula que define `ROOT` e `os.environ["WYSCOUT_*"]`; usa o Python da venv `apps/api` (onde estão `pandas` / `pyarrow`).

In [ ]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "download_heatmaps.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz do repo: cd apps/api && uv sync"
    )
subprocess.run([str(venv_python), str(script)], check=True, cwd=ROOT)

## 2. Limpeza — `cleaning_data.py`

Lê/escreve UTF-8 em `data/players/wyscout/*.csv` e normaliza células que parecem listas Python (ex.: `['LWF', 'LW']` → texto separado por vírgulas).

In [ ]:
%run scripts/cleaning_data.py

## 2.5. Enriquecimento Transfermarkt — `enrich_with_tm.py`

Corre **depois** da limpeza (passo 2). Usa `data/tm/people.csv` e `data/tm/players.csv` para juntar `image_url` e altura (`height_in_cm` → coluna `Height`) aos CSV Wyscout em `data/players/wyscout/` (sobrescreve in-place).

- Primeira execução: `--dry-run` (preview; não grava).
- Segunda execução: sem `--dry-run` para gravar.

In [ ]:
%run ./scripts/enrich_with_tm.py 

## 3. New performance Index — `new_performance_index.py`


In [ ]:
%run  ../transformation/new_performance_index.py

## 4. CSV → Parquet — `csv_to_parquet.py`

Converte cada `*_all_leagues.csv` em `data/players/all/` para `.parquet` (snappy). Os CSV agregados por época precisam de já existir nessa pasta (se os geras noutro script, corre esse passo antes deste).

In [ ]:
%run ../scripts/csv_to_parquet.py

## 5. Update potential scores— `train_potential.py`

In [ ]:
%run scripts/train_potential.py

## check duplicates

In [ ]:
%run scripts/check_wyscout_duplicate_players.py --only-with-dups


## Remover duplicados nos CSV Wyscout — `dedupe_wyscout_csv_rows.py`

Por ficheiro em `data/players/wyscout/`, remove linhas repetidas com a mesma chave **(Wyscout id, nome do jogador, equipa)**. Por omissão mantém a **última** ocorrência (`--keep last`); usa `--keep first` ou `--dry-run` no terminal conforme precises.

**Ordem lógica:** corre isto logo **após** a limpeza (secção 2) e **antes** do índice de performance (secção 3). Esta célula está no fim do notebook só como referência ao script; sobe-a ou corre-a no momento certo do pipeline.

In [ ]:
%run scripts/dedupe_wyscout_csv_rows.py

## 10. Filtered player valuations — `build_player_valuations.py`

Cria `data/tm/player_valuations_filtered.csv` (e parquet com `--parquet`) com colunas  
`wyscout_id`, `key_transfermarkt`, `date`, `market_value_in_eur`. Por defeito mantém  
qualquer jogador que apareça em pelo menos um ficheiro Wyscout (`--mode union`).

_Para o xTV v2 e `utils.tm_market_value` usa-se o ficheiro bruto **`player_valuations.csv`** — este filtro é opcional (só reduz I/O noutros fluxos)._

_Para uso estrito (jogadores em **todos** os ficheiros — geralmente devolve 0):  
`--mode intersection`._


In [ ]:
%run build_player_valuations.py --parquet

## 11. Club logos parquet — `build_club_logos.py`

Percorre todos os CSVs Wyscout e produz `data/teams/club_logos.parquet` com colunas **`team`**, **`logo_url`**, **`competition`** (doméstica; alinha com `league` nos parquets), **`n_seasons`**, **`latest_file`**. Inclui todas as equipas (sem filtro). Se o CSV não tiver coluna `Competition`, fica string vazia `""`.

In [ ]:
%run build_club_logos.py

## 12. Resumo — ordem sugerida de scripts

1. **`download_data.py`** — CSV Wyscout por liga/temporada em `data/players/wyscout/`.
2. **`download_heatmaps.py`** — heatmaps TM (venv API; mesmo env Wyscout das células iniciais).
3. **`cleaning_data.py`** — limpezas/fixes antes do pipeline estatístico.
4. **`enrich_with_tm.py`** — foto/altura desde `people.csv` + `players.csv`.
5. **`transformation/new_performance_index.py`** — gera `{ano}_all_leagues.csv`.
6. **`csv_to_parquet.py`** — `data/players/all/{ano}_all_leagues.parquet` (consumo pela API DuckDB).
7. **`train_potential.py`** — scores de potencial (parquet esperado pela API).
8. **`check_wyscout_duplicate_players.py`** + **`dedupe_wyscout_csv_rows.py`** — auditoria/remoção de duplicados nos CSV Wyscout antes de novo ciclo opcional de (5–6).
9. **`build_player_valuations.py --parquet`** — subset TM filtrado por jogadores Wyscout (`player_valuations_filtered`).
10. **`build_club_logos.py`** — `club_logos.parquet` (`team`, `logo_url`, `competition`).
11. **`train_xtv.py`** — modelo xTV v2 (transferências TM × parquets Wyscout; precisa `data/tm/transfers.csv`, `people.csv`, `players.csv`, `player_valuations.csv`, `clubs.csv`, parquets em `data/players/all/`). Grava **`models/xtv_v2.joblib`**, **`models/xtv_v2.json`**, **`data/tm/xtv_id_mapping.parquet`** e prior de destino em `data/tm/` (venv com sklearn/joblib/scipy).
12. **`apply_xtv_to_parquets.py`** — acrescenta **`tm_market_value_eur`** e **`x_tv_eur`** nos parquets (por defeito **sobrescreve** `{ano}_all_leagues.parquet`; `--sidecar` grava `*_xtv.parquet` à parte).

_Dependências de dados_: TM (`data/tm/…`) atualizadas por ingestão própria; este notebook não faz scrape TM.

## 13. xTV v2 — treino (`train_xtv.py`)

Junta **`transfers.csv`** aos parquets **`{ano}_all_leagues.parquet`**, mapeia IDs Wyscout↔TM (`xtv_id_mapping.parquet`) e usa **`player_valuations.csv`** (MV as-of na data da transferência). Treina duas cabeças (ratio `log(fee/mv)` quando há MV; `log(fee)` absoluto) com split temporal em **`--cutoff`** (default `2023-07-01`); no fim refit no dataset completo.

- Saídas: **`models/xtv_v2.joblib`** + **`models/xtv_v2.json`** (métricas), **`data/tm/xtv_id_mapping.parquet`**, prior de destino em `data/tm/`.
- Argumentos úteis: `--max-rows N` (debug), `--fuzzy-threshold 92`, `--no-refresh-mappings` (reutilizar mapping existente).

Usa o **mesmo Python do venv da API** (sklearn, joblib, scipy).

In [ ]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "train_xtv.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz: cd apps/api && uv sync",
    )

subprocess.run(
    [
        str(venv_python),
        str(script),
        # "--cutoff", "2023-07-01",  # default na CLI; descomenta para outro split
        # "--max-rows", "8000",
        # "--no-refresh-mappings",
    ],
    check=True,
    cwd=ROOT,
)


## 14. xTV v2 — aplicar aos parquets (`apply_xtv_to_parquets.py`)

Lê **`models/xtv_v2.joblib`**, acrescenta **`tm_market_value_eur`** (MV TM as-of fim de época; imputação por peers se em falta) e **`x_tv_eur`** (média de previsões com destinos amostrados do prior de treino).

- Por defeito **sobrescreve** `{ano}_all_leagues.parquet` (o que a API DuckDB lê).
- **`--sidecar`** grava `{ano}_all_leagues_xtv.parquet` sem tocar no original.
- **`--dry-run`** lista ficheiros sem gravar.

In [ ]:
import subprocess

venv_python = ROOT / "apps" / "api" / ".venv" / "bin" / "python"
script = ROOT / "scripts" / "apply_xtv_to_parquets.py"
if not venv_python.is_file():
    raise FileNotFoundError(
        f"Falta {venv_python} — na raiz: cd apps/api && uv sync",
    )

subprocess.run(
    [
        str(venv_python),
        str(script),
        "--model",
        str(ROOT / "models" / "xtv_v2.joblib"),
        # "--sidecar",             # grava *_all_leagues_xtv.parquet em vez de overwrite
        # "--dry-run",             # imprime apenas o que corraria
    ],
    check=True,
    cwd=ROOT,
)
